# Code Model Capabilities

This notebook deep-dives the capability map of modern coding models: completion, generation, explanation, repair, refactoring, test generation, and multi-file reasoning. You will implement small harnesses that mirror how products evaluate these skills.

```mermaid
mindmap
  root((Code Capabilities))
    Completion
      FIM
      Next line
    Generation
      Functions
      Services
    Understanding
      Summaries
      Q&A
    Transformation
      Repair
      Refactor
      Migrate
    Verification
      Tests
      Types
```


## Learning Objectives

- Enumerate and contrast primary coding capabilities
- Explain FIM vs left-to-right chat generation
- Design a repair-loop harness shape
- Prompt for high-signal test generation
- Apply a multi-file reasoning checklist


## 1. Capability Map

| Capability | Definition | Why it matters | Failure mode |
|------------|------------|----------------|--------------|
| Completion & FIM | Predict span given surroundings | Core IDE UX | Wrong scope / stale APIs |
| NL → code | Implement from description | Greenfield speed | Spec underspecified |
| Code → NL | Explain / document | Onboarding | Hallucinated intent |
| Repair | Fix failing tests/logs | Debugging leverage | Patch wrong root cause |
| Refactor / migrate | Structure-preserving change | Large-scale change | Behavioral drift |
| Test generation | Produce tests | Regression safety | Testing implementation not contract |
| Multi-file | Cross-module edits | Real repos | Missed dependents |

### When to use which
- **Completion:** typing in a known pattern
- **NL→code:** new units with clear contracts
- **Repair:** failing CI with reproducible tests
- **Migrate:** mechanical API changes + validation suite


## 2. Fill-in-the-Middle (FIM) vs Chat Generation

### Definition
**FIM** conditions on both prefix and suffix so the model fills a hole—ideal for cursor-middle edits. **Chat generation** is usually left-to-right from a conversational prompt.

### Intuition
FIM is like Mad Libs for code: the surrounding structure constrains the answer. Chat is like writing a new scene from a brief.

### Pitfalls
- Provider FIM token formats differ (`<fim_prefix>`, etc.)
- Chat models may ignore suffix constraints unless you structure the prompt carefully


In [ ]:
# Demo 1 — FIM prompt structure (formats vary by provider)
def build_fim_prompt(prefix: str, suffix: str, style: str = "generic") -> str:
    if style == "generic":
        return (
            "<fim_prefix>" + prefix +
            "<fim_suffix>" + suffix +
            "<fim_middle>"
        )
    if style == "chat_emulation":
        return (
            "Complete the missing middle. Do not repeat prefix/suffix.\n\n"
            f"PREFIX:\n{prefix}\n\nSUFFIX:\n{suffix}\n\nMIDDLE:\n"
        )
    raise ValueError(style)

prefix = "def add(a: int, b: int) -> int:\n    "
suffix = "\n\ndef add3(x):\n    return add(x, 3)\n"
print(build_fim_prompt(prefix, suffix))
print("---")
print(build_fim_prompt(prefix, suffix, style="chat_emulation")[:200], "...")


In [ ]:
# Demo 2 — Rank completion candidates with simple heuristics
import ast

def candidate_ok(prefix: str, middle: str, suffix: str) -> tuple[bool, str]:
    src = prefix + middle + suffix
    try:
        ast.parse(src)
    except SyntaxError as e:
        return False, f"syntax: {e}"
    if "TODO" in middle:
        return False, "contains TODO"
    return True, "ok"

prefix = "def area(r):\n    "
suffix = "\n"
cands = ["return 3.14 * r * r\n", "return\n", "TODO\n"]
for c in cands:
    print(repr(c), "->", candidate_ok(prefix, c, suffix))


## 3. Repair Loop Pattern

### How it works
```
failing test / stacktrace
        │
        v
   localize (file/symbol)
        │
        v
   propose patch
        │
        v
   run tests ──fail──► update context ──► propose again
        │
       pass
```

### Pitfalls
- Infinite loops on flaky tests
- Overfitting to a single failing assertion
- Editing tests to “make green” without approval


In [ ]:
# Demo 3 — Toy repair loop harness (no live model)
from dataclasses import dataclass, field

@dataclass
class Attempt:
    patch: str
    passed: bool
    log: str

@dataclass
class RepairState:
    goal: str
    attempts: list[Attempt] = field(default_factory=list)
    max_attempts: int = 3

def mock_model_propose(state: RepairState) -> str:
    # pretend model improves each try
    n = len(state.attempts)
    return f"# patch attempt {n+1}\nassert True  # fixed logic\n"

def mock_run_tests(patch: str) -> Attempt:
    passed = "attempt 3" in patch or "fixed logic" in patch and "attempt 2" in patch
    # succeed on 2nd attempt for demo
    passed = "attempt 2" in patch
    return Attempt(patch, passed, "OK" if passed else "AssertionError")

def repair_loop(goal: str) -> RepairState:
    state = RepairState(goal=goal)
    while len(state.attempts) < state.max_attempts:
        patch = mock_model_propose(state)
        result = mock_run_tests(patch)
        state.attempts.append(result)
        if result.passed:
            break
    return state

st = repair_loop("fix off-by-one in pager")
print([(a.passed, a.log, a.patch.splitlines()[0]) for a in st.attempts])


## 4. Test Generation Capability

### Why it matters
Tests are the cheapest durable oracle for coding models. Good test generation encodes the **contract**, not the current buggy implementation.

### When to use
- Characterization tests before refactor
- Property-based seeds for parsers
- API contract tests from OpenAPI

### Pitfalls
- Brittle tests tied to private internals
- Asserting on hallucinated exception types
- Skipping adversarial / security cases


In [ ]:
# Demo 4 — Prompt template + lightweight test quality checks
import ast

TEST_GEN_TEMPLATE = '''
Given this function under test:

```python
{code}
```

Write pytest tests that cover:
1) happy path 2) edge cases 3) invalid inputs
Rules: no network, deterministic, test public behavior only.
'''

def test_quality(src: str) -> dict:
    tree = ast.parse(src)
    asserts = sum(isinstance(n, ast.Assert) for n in ast.walk(tree))
    functions = [n.name for n in ast.walk(tree) if isinstance(n, ast.FunctionDef) and n.name.startswith("test_")]
    return {"n_tests": len(functions), "n_asserts": asserts, "names": functions}

sample_tests = '''
def test_empty():
    assert normalize("") == ""

def test_basic():
    assert normalize("A") == "a"
'''
print(TEST_GEN_TEMPLATE.format(code="def normalize(s):\n    return s.strip().lower()\n")[:180], "...")
print(test_quality(sample_tests))


## 5. Multi-file Reasoning Checklist

Use this whenever a task spans modules:

1. **Locate entrypoints** — CLI, HTTP handlers, jobs
2. **Trace data flow** — request → domain → DB/API
3. **Find dependents** — who imports the symbol?
4. **Identify contracts** — types, schemas, events
5. **Plan edit set** — minimal files + tests
6. **Migration strategy** — feature flag / dual-write if needed
7. **Verification** — unit + integration + rollout

```ascii
svc/api.py ──► svc/service.py ──► svc/repo.py
     │                │
     └──── tests ─────┘
```


In [ ]:
# Demo 5 — Import graph sketch for multi-file planning
import ast
from pathlib import Path

def imports_in(source: str) -> set[str]:
    tree = ast.parse(source)
    out = set()
    for n in ast.walk(tree):
        if isinstance(n, ast.Import):
            for a in n.names:
                out.add(a.name.split(".")[0])
        elif isinstance(n, ast.ImportFrom) and n.module:
            out.add(n.module.split(".")[0])
    return out

files = {
    "api.py": "from service import create_user\n",
    "service.py": "from repo import save\n",
    "repo.py": "import json\n",
}
graph = {f: imports_in(src) & set(x[:-3] for x in files) for f, src in files.items()}
print(graph)


### Try it yourself — Capabilities

- Convert a chat prompt into an equivalent FIM prompt for a real file hole
- Implement a repair loop that stops on repeated fingerprints
- Generate tests for a function, then mutate the function and see which tests catch it
- Draw an import graph for a small package and plan a rename migration


## Glossary / Key Terms

| Term | Meaning |
|------|--------|
| `FIM` | Fill-in-the-middle generation |
| `Repair loop` | Iterative patch→test cycle |
| `Characterization test` | Test locking current behavior before change |
| `Oracle` | Trusted checker of correctness (tests, types, human) |


## Deep Dive Workshop — 02 Code Model Capabilities

This section expands the notebook into instructor/textbook depth. Work through each subsection: **definition → why it matters → how it works → intuition → pitfalls → when to use**.

```mermaid
flowchart TB
  D[Definition] --> W[Why it matters]
  W --> H[How it works]
  H --> I[Intuition]
  I --> P[Pitfalls]
  P --> U[When to use]
```


### Concept card pack for `02-code-model-capabilities`

| Concept | Definition | Why it matters | Common pitfall |
|---------|------------|----------------|----------------|
| Primary abstraction | Core object this lesson centers on | Anchors design conversations | Vague naming |
| Quality oracle | How you know the system is right | Prevents demo-driven development | Using vibes only |
| Latency budget | Max user-visible wait | Drives architecture | Ignoring TTFT vs e2e |
| Cost unit | $ per successful task | Makes tradeoffs real | Optimizing tokens not outcomes |
| Trust boundary | Where data/control changes hands | Security design | Treating vendors as internal |
| Feedback loop | How production improves the system | Sustainable quality | No path from thumbs-down to evals |

**Intuition:** If you cannot fill this table for your system, you are not ready to choose models or frameworks.


### Pipeline walkthrough (apply to 02-code-model-capabilities)

```
1. Input arrives (user / job / webhook)
2. Normalize + authorize + budget check
3. Gather context (files, RAG, tools, memory)
4. Model / deterministic compute
5. Validate output (schema, policy, tests)
6. Side effects (write, ticket, PR) with authz
7. Observe (metrics, traces, feedback)
8. Learn (eval suite growth, prompt/model revision)
```

**When to compress steps:** tiny internal tools. **When to keep all steps:** multi-tenant or regulated production.


### Coding-models advanced notes

**Fill-in-the-middle formats** differ by vendor; always keep an adapter layer.  
**Repo agents** should treat tests as the north star and protect test files by default.  
**Coding RAG** should combine symbol lookup + BM25 + embeddings; embeddings alone miss identifiers.

| Task | Prefer | Avoid |
|------|--------|-------|
| Ghost text | FIM-capable small/fast model | Giant chat model sync |
| API migration | Agent + tests | Single-shot whole-repo rewrite |
| Explain legacy | Chat + citations | Uncited summaries |


In [ ]:
# Extra demo — diff extraction toy for coding assistants
import re

def extract_fenced_blocks(text: str) -> list[tuple[str, str]]:
    pat = re.compile(r"```(\w+)?\n(.*?)```", re.S)
    return [(m.group(1) or 'txt', m.group(2)) for m in pat.finditer(text)]

sample = '''Here is a fix:\n```python\ndef add(a,b):\n    return a+b\n```\n'''
print(extract_fenced_blocks(sample))


In [ ]:
# Extra demo — simple symbol index for coding RAG
import ast
from collections import defaultdict

def index_symbols(source: str, path: str) -> dict[str, list[str]]:
    tree = ast.parse(source)
    idx = defaultdict(list)
    for n in ast.walk(tree):
        if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            idx[n.name].append(f"{path}:{n.lineno}")
    return dict(idx)

print(index_symbols('class Foo:\n  def bar(self):\n    pass\n', 'a.py'))


### Sample interview Q&A — coding models

**Q:** Copilot-quality inline completion is slow. What do you do?  
**A:** Separate completion model from chat model; shrink context to locals + imports; consider speculative decoding / smaller quantized model; measure acceptance rate not just tok/s.

**Q:** How do you evaluate a coding assistant for a monorepo?  
**A:** Private suite: completion acceptance, unit-test pass on generated patches, security scanner findings, and human review on a stratified sample of PR diffs.


### Comparison matrix exercise

Fill this for two competing designs in this topic:

| Dimension | Option A | Option B | Winner / why |
|-----------|----------|----------|--------------|
| Latency | | | |
| Cost at 10× scale | | | |
| Quality risk | | | |
| Ops burden | | | |
| Security / privacy | | | |
| Time to MVP | | | |


In [ ]:
# Workshop demo — decision scorecard
from dataclasses import dataclass

@dataclass
class Option:
    name: str
    latency: int  # 1=best .. 5=worst
    cost: int
    quality_risk: int
    ops: int
    security: int

def score(o: Option, weights=None) -> float:
    weights = weights or dict(latency=1, cost=1, quality_risk=2, ops=1, security=2)
    return (
        o.latency*weights['latency'] + o.cost*weights['cost'] +
        o.quality_risk*weights['quality_risk'] + o.ops*weights['ops'] +
        o.security*weights['security']
    )

a = Option('A', 2, 3, 2, 2, 2)
b = Option('B', 3, 1, 3, 4, 2)
print(a.name, score(a), b.name, score(b), '-> prefer', a.name if score(a)<score(b) else b.name)


In [ ]:
# Workshop demo — experiment log (use while studying this notebook)
from dataclasses import dataclass, asdict
import json, time

@dataclass
class Experiment:
    hypothesis: str
    setup: str
    metric: str
    baseline: float | None = None
    treatment: float | None = None
    notes: str = ''
    ts: float = 0.0

    def __post_init__(self):
        if not self.ts:
            self.ts = time.time()

exp = Experiment(
    hypothesis='Technique from this lesson improves the primary metric',
    setup='Describe fixtures / model / dataset version',
    metric='name of metric',
    baseline=0.0,
    treatment=0.0,
)
print(json.dumps(asdict(exp), indent=2))


### ASCII architecture sketch template

```
[ Clients ]
     |
[ Edge / API Gateway ] -- authn/z, rate limit
     |
[ Orchestration ] ------+-- prompts / policies
     |                  +-- eval hooks
     +-- context layer (RAG / tools / memory)
     |
[ Model interface ] ---- local and/or cloud
     |
[ Data plane ] --------- indexes, OLTP, object store
     |
[ Observability ] ------ logs, metrics, traces, feedback
```

Copy into your notes and annotate trust boundaries with `***`.


### Pitfalls clinic (read aloud)

1. **Metric theater** — optimizing a proxy that users don't feel  
2. **Context stuffing** — more tokens ≠ more truth  
3. **Prompt as security** — never the only control  
4. **Hidden coupling** — tools/models/indexes version-drift  
5. **No rollback** — can't revert prompt/model quickly  
6. **Eval contamination** — testing on training-like snippets  
7. **Happy-path demos** — skipping adversarial & empty-retrieve cases  


### Try it yourself — extended set

1. Teach the top 3 ideas from this notebook to a rubber duck in 5 minutes  
2. Write 5 quiz questions (with answers) for a junior engineer  
3. Implement one code demo with a real dependency (API or local model) using env placeholders  
4. Break a naive design on purpose; list the failure mode and the fix  
5. Add two rows to your personal glossary with examples from work  
6. Produce a one-page cheat sheet you could use in an interview  


### Mini case study

**Scenario:** Leadership wants this capability in production in six weeks with two engineers.

**Your job:** Propose an MVP that keeps irreversible risks controlled, names the eval gates, and lists what you explicitly defer.

Deliverable structure:
- MVP user story  
- Non-goals  
- Architecture (6 boxes max)  
- Eval gate table  
- Risk register (top 5)  
- Week-by-week plan  


In [ ]:
# Case study helper — risk register
import pandas as pd

risks = pd.DataFrame([
    {'risk': 'quality_miss', 'likelihood': 3, 'impact': 3, 'mitigation': 'golden evals + canary'},
    {'risk': 'cost_overrun', 'likelihood': 3, 'impact': 2, 'mitigation': 'budgets + cache'},
    {'risk': 'data_leak', 'likelihood': 2, 'impact': 5, 'mitigation': 'ACL + redaction'},
    {'risk': 'prompt_injection', 'likelihood': 4, 'impact': 4, 'mitigation': 'boundaries + allowlists'},
    {'risk': 'ops_pages', 'likelihood': 3, 'impact': 3, 'mitigation': 'runbooks + rollback'},
])
risks['score'] = risks.likelihood * risks.impact
print(risks.sort_values('score', ascending=False).to_string(index=False))


### Interview drill (topic-local)

Use the STAR or design template. Timebox 8 minutes.

**Prompt:** “Walk me through how you would productionize the main idea of this notebook.”

Checklist for a strong answer:
- [ ] Clarifying questions  
- [ ] Constraints & numbers  
- [ ] Diagram  
- [ ] Deep dive on hardest part  
- [ ] Evals  
- [ ] Security  
- [ ] Rollout / rollback  


### Glossary boost

| Term | Expanded meaning |
|------|------------------|
| Canary | Partial traffic to a new variant with automatic rollback |
| Golden set | Versioned labeled examples for regression |
| TTFT | Time to first token — interactive UX driver |
| Packing | Selecting/ordering context under a token budget |
| HITL | Human approval inserted before side effects |
| Idempotency | Safe retries without duplicate side effects |
| Shadow traffic | New system sees traffic but doesn't affect users |
| Circuit breaker | Stop calling a failing dependency temporarily |


In [ ]:
# Self-check quiz (run and answer mentally before printing answers)
QUESTIONS = [
    'What oracle proves success for this topic?',
    'Name one metric that can be gamed and a better alternative.',
    'What is the top security failure mode?',
    'What would you defer in an MVP?',
    'How do you rollback a bad change here?',
]
for i, q in enumerate(QUESTIONS, 1):
    print(f'Q{i}. {q}')
print('\n--- suggested answer hints ---')
HINTS = [
    'executable tests / task success / human rubric',
    'longer answers != better; use task success',
    'trust boundary crossing / injection / ACL',
    'multi-agent, perfect UI, every connector',
    'versioned prompts/models + traffic switch',
]
for h in HINTS:
    print('-', h)


### Further practice roadmap for `02-code-model-capabilities`

| Horizon | Action |
|---------|--------|
| Today | Re-run all code cells; note questions |
| This week | Apply one technique to a real repo/service |
| This month | Add an eval or security test covering this topic |
| Interview ready | Give a 10-minute teach-back with a diagram |


## Lab: End-to-end scenario

Work this scenario in your notes, then implement the smallest possible spike.

### Scenario brief
A team wants to adopt the techniques from this notebook for a **real internal tool** used daily by 200 people. Leadership cares about reliability and auditability more than flashy demos.

### Deliverables
1. One-paragraph problem statement  
2. Success metrics (3) with oracles  
3. Architecture sketch with trust boundaries  
4. Threats / failure modes (5)  
5. Eval plan (offline + online)  
6. 2-week MVP scope and explicit non-goals  

### Review questions
- What happens when context is empty?  
- What happens when the model is down?  
- What happens when a user is malicious?  
- How do you prove a release is safer/better than last week?  


In [ ]:
# Lab helper — MVP scope tracker
from dataclasses import dataclass, field

@dataclass
class MVP:
    must: list[str] = field(default_factory=list)
    should: list[str] = field(default_factory=list)
    defer: list[str] = field(default_factory=list)

    def show(self):
        for label, items in [('MUST', self.must), ('SHOULD', self.should), ('DEFER', self.defer)]:
            print(label)
            for i in items:
                print(' -', i)

mvp = MVP(
    must=['core happy path', 'authn', 'basic eval smoke', 'rollback switch'],
    should=['streaming UX', 'dashboards'],
    defer=['multi-agent', 'perfect personalization', 'every connector'],
)
mvp.show()


## Operator runbook sketch

| Symptom | Likely cause | First checks | Mitigation |
|---------|--------------|--------------|------------|
| Latency spike | Downstream model / retrieve | p95 by stage, saturation | shed load, failover |
| Quality drop | Prompt/model/index change | diff versions, eval slice | rollback |
| Cost spike | loops / huge prompts | tokens/req, step counts | budget breaker |
| Security alert | injection / ACL | traces + retrieved IDs | kill switch |

Keep this table in your ops wiki; customize per system.


In [ ]:
# Operator helper — stage latency rollup
from statistics import mean

stages = {
    'gateway': [20, 25, 22],
    'retrieve': [80, 120, 95],
    'generate': [900, 1100, 980],
}
for k, v in stages.items():
    print(f'{k:10} mean={mean(v):.0f}ms max={max(v)}ms')
print('e2e~', sum(mean(v) for v in stages.values()), 'ms')


## Teaching notes (for study groups)

- Start with the comparison table; argue both sides for 5 minutes  
- Pair-program one demo cell with a real endpoint (placeholder keys)  
- Each person writes one failure case the suite must catch  
- End with a 60-second summary of when *not* to use the technique  


## Summary & Key Takeaways

- Capabilities span complete→generate→repair→multi-file
- FIM and chat are different conditioning formats
- Repair needs harnesses, budgets, and stop conditions
- Test generation should encode contracts
- Multi-file work needs explicit dependency reasoning
